In [ ]:
_SEED = 0

#### Basic package ####
import copy
import datetime
import functools
import glob
import importlib
import itertools
import os
import pickle
import random
import re
import shutil
import subprocess
import sys
import time
import typing
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

from pprint import pprint
from tqdm import tqdm

#### File formats ####
import json
import yaml
import toml
import sqlite3
import h5py
import gzip

#### Math package ####
import math
import numpy as np
import scipy.spatial
import scipy.special
import scipy.stats
import numba

np.set_printoptions(suppress=True)

#### Data operation package ####
import pandas as pd
import polars as pl

os.environ["POLARS_FMT_MAX_ROWS"] = "20"

#### Parallel package ####
import dask
import dask.array as da
import dask.bag as db
import dask.dataframe as dd
import distributed
import joblib
from joblib import Parallel, delayed

#### Machine learning package ####

import sklearn.cluster
import sklearn.decomposition
import sklearn.linear_model
import sklearn.manifold
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing

#### Deep learning package ####

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchmetrics
import lightning as L


#### Visualization package ####
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import patches as mpl_patch
from matplotlib import rcParams
from matplotlib.ticker import FuncFormatter
from matplotlib_venn import venn2, venn2_circles, venn2_unweighted, venn3, venn3_circles, venn3_unweighted
import seaborn as sns

rcParams["font.sans-serif"] = ["Arial"]
rcParams["font.size"] = 7.5
rcParams["figure.dpi"] = 150



MainDir = Path(".").resolve()

sys.path.append(str(MainDir))
import utils

sys.path.append(str(Path(".").resolve().parent.parent.joinpath("LiPAna")))
import lipana

#### Seed ####
random.seed(_SEED)
np.random.seed(_SEED)
L.seed_everything(_SEED)



Seed set to 0


0

In [ ]:
#### Work space ####
ProtDataDir = MainDir.joinpath("searches")
DownloadDataDir = MainDir.joinpath("Data-Download")
SeqDataDir = MainDir.joinpath("Data-Sequence")
CLDataDir = MainDir.joinpath("Data-CrossLab")
ProjectSrcDir = MainDir.joinpath("project_src")
ntpep_model_dir = MainDir.joinpath("ntpep_attr_model")
NotebookOutputDir = MainDir.joinpath("IPyNotebook", "NotebookOutput")
NotebookTempDir = MainDir.joinpath("IPyNotebook", "_temp")
print("MainDir:", MainDir)
print("ProtDataDir:", ProtDataDir)
print("DownloadDataDir:", DownloadDataDir)
print("SeqDataDir:", SeqDataDir)
print("CLDataDir:", CLDataDir)
print("ProjectSrcDir:", ProjectSrcDir)
print("ntpep_model_dir:", ntpep_model_dir)
print("NotebookOutputDir:", NotebookOutputDir)
print("NotebookTempDir:", NotebookTempDir)

if os.name == "nt":
    rcParams["font.family"] = "sans-serif"  # 'sans-serif' in windows, 'Liberation Sans' in linux
elif os.name == "posix":
    rcParams["font.family"] = "Liberation Sans"  # 'sans-serif' in windows, 'Liberation Sans' in linux
  


workspace = ProtDataDir.joinpath("LiP_DIANN192")
diann_report_dir_1 = ProtDataDir.joinpath("HY_LiP_DDA_DIA")
notebook_date = "26adjust"
logger = utils.console_file_logger("lipana")




In [ ]:

search_reports_paths = {
    "OR-DDALib": diann_report_dir_1.joinpath(
        "example_diann_report", "report_filtered.tsv",
    ),
}
search_reports_paths = {f"DIANN-{k}": v for k, v in search_reports_paths.items()}

utils.check_path_in_dict(search_reports_paths, shown_filename_right_idx=2)

parsed_fasta = lipana.parse_fasta(
    fasta_path=MainDir.joinpath(
        "fastas",
        "test.fasta",
    ),
    contam_fasta_path=MainDir.joinpath("fastas", "contaminants_type12.fasta"),
    workspace=workspace,
    resume=True,
    write_parsed_fasta=True,
)

exp_info_path = MainDir.joinpath("experiment_info_adjust_factor.txt")
exp_layout = lipana.ExperimentLayout.from_file(exp_info_path)



In [ ]:
search_reports = {}

for name, path in search_reports_paths.items():
    if "DIANN" not in name:
        continue

    report = lipana.DIANNReport.load_search_report(
        path,
        exp_layout=exp_layout,
        parsed_fasta=parsed_fasta,
        do_species_annotation=True,
        pre_annotation_filter=(
            (pl.col("Q.Value") < 0.01)
            & (pl.col("Lib.PG.Q.Value") < 0.01)
            & (pl.col("Protein.Group").is_not_null())
            & (pl.col("Precursor.Quantity").is_not_null())
            & (pl.col("Precursor.Quantity") > 1.1)
        ),
        post_annotation_filter=None,
        restricted_cut_sites=("K", "R"),
        expand_to_cut_site_level=True,
        resume=False,
        write_processed_report=True,
        processed_report_filename_suffix="-processed.parquet",
        batch_size=10_000,
        n_jobs=-1,
    )
    search_reports[name] = report

for name, path in search_reports_paths.items():
    if "SN" not in name:
        continue

    report = lipana.SpectronautReport.load_search_report(
        path,
        exp_layout=exp_layout,
        parsed_fasta=parsed_fasta,
        do_species_annotation=True,
        pre_annotation_filter=(
            (pl.col("PG.ProteinGroups").is_not_null())
            & (pl.col("FG.Quantity").is_not_null())
            & (pl.col("FG.Quantity") > 1.1)
        ),
        post_annotation_filter=None,
        restricted_cut_sites=("K", "R"),
        expand_to_cut_site_level=True,
        resume=False,
        write_processed_report=True,
        processed_report_filename_suffix="-processed.parquet",
        batch_size=10_000,
        n_jobs=-1,
    )
    search_reports[name] = report

parsed_fasta.dump()



In [ ]:
report_filters = {
    "All": True,
    "AllEntry": True,
    None: True,
    "HUMAN": pl.col("mapped_species_from_peptide") == "HUMAN",
    "YEAST": pl.col("mapped_species_from_peptide") == "YEAST",
    "Fully": pl.col("peptide_enzymatic_specificity") == "fully_specific",
    "Semi": pl.col("peptide_enzymatic_specificity") == "semi_specific",
    "KR-end": pl.col("peptide_c_term_aa").is_in(("K", "R")),
    "Non-KR-end": (~pl.col("peptide_c_term_aa").is_in(("K", "R"))),
    "non-restricted": (~pl.col("cut_site_is_restricted")),
    "restricted": pl.col("cut_site_is_restricted"),
    "restricted from fully pep": (
        (pl.col("peptide_enzymatic_specificity") == "fully_specific") & pl.col("cut_site_is_restricted")
    ),
    "restricted from semi pep": (
        (pl.col("peptide_enzymatic_specificity") == "semi_specific") & pl.col("cut_site_is_restricted")
    ),
    ####
    **utils.union_dicts(
        *[
            {
                f"All-Rep>={n}": (
                     (pl.col("H2_Y100_detected_reps") >= n)
                    | (pl.col("H5_Y100_detected_reps") >= n)
                    | (pl.col("H30_Y100_detected_reps") >= n)
                ),
                f"H2_Y100-Rep>={n}": (pl.col("H2_Y100_detected_reps") >= n),
                f"H5_Y100-Rep>={n}": (pl.col("H5_Y100_detected_reps") >= n),
                f"H30_Y100-Rep>={n}": (pl.col("H30_Y100_detected_reps") >= n),
            }
            for n in (1, 2, 3, 4)
        ]
    ),
    **utils.union_dicts(
        *[
            {
                f"All-CV<={t}": (
                     (pl.col("H2_Y100_cv_2reps") <= t)
                    | (pl.col("H5_Y100_cv_2reps") <= t)
                    | (pl.col("H30_Y100_cv_2reps") <= t)
                ),
                f"H2_Y100-CV<={t}": (pl.col("H2_Y100_cv_2reps") <= t),
                f"H5_Y100-CV<={t}": (pl.col("H5_Y100_cv_2reps") <= t),
                f"H30_Y100-CV<={t}": (pl.col("H30_Y100_cv_2reps") <= t),
            }
            for t in (30, 20, 15, 10)
        ]
    ),
}

primary_entries = [
    ("protein_group", "AllEntry"),

]
base_quants_group = [
    lipana.cm.precursor_quantity_ms2,
    lipana.cm.precursor_quantity_ms2_normalised,
    (lipana.cm.precursor_quantity_ms2, lipana.cm.precursor_quantity_ms1),
    (lipana.cm.precursor_quantity_ms2_normalised, lipana.cm.precursor_quantity_ms1_normalised),
]
reported_quantification_groups = {
    "protein_group": (
        "PG.Quantity",
    ),
    "stripped_peptide": ("PEP.Quantity",),
    "modified_peptide": ("EG.TotalQuantity (Settings)",),
    "precursor": (
        "precursor_quantity",
        "precursor_quantity_normalised",
        "precursor_quantity_ms1",
        "precursor_quantity_ms1_normalised",
        "precursor_quantity_ms2",
        "precursor_quantity_ms2_normalised",
    ),
}



In [ ]:
for name, report in search_reports.items():
    for primary_entry, used_enzyme_type in primary_entries:
        for quant_method in ("maxlfq", ):
            for base_quants in base_quants_group:
                quant_data = report.construct_and_attach_quant_data(
                    quant_name=f"{quant_method}-{base_quants if isinstance(base_quants, str) else '-'.join(base_quants)}-{used_enzyme_type}",
                    method=quant_method,
                    filter_condition=(
                        (~pl.col("mapped_species_from_peptide").str.contains(";", literal=True))
                        & report_filters[used_enzyme_type]
                    ),
                    run_col=lipana.cm.run,
                    primary_entry_col=primary_entry,
                    low_level_entry_col=lipana.cm.precursor,
                    base_quant_col=base_quants,
                    require_expansion=False,
                    concat_entry_after_expansion=None,
                    remove_below_threshold=1.1,
                    quant_input_name=None,
                    attach_quant_input=False,
                )
                quant_data.calc_cv(cond="all", min_reps=3, temp_reverse_log_scale=2)
                quant_data.calc_cv(cond="all", min_reps=2, temp_reverse_log_scale=2)
                quant_data.count_detected_replicates()
                quant_data.calc_ratio(
                    base_cond="H5_Y100",
                    is_log=True,
                    temp_reverse_log_scale=2,
                    div_method="agg_and_divide",
                    agg_method="mean",
                )

    for entry_name, quants in reported_quantification_groups.items():
        for quant in quants:
            if quant in report.df.columns:
                quant_data = report.attach_quant_data(
                    lipana.convert_long_report_to_wide(
                        report,
                        index_col=entry_name,
                        column_col="run",
                        value_col=quant,
                        do_log_scale=2,
                        pl_filter=(~pl.col("mapped_species_from_peptide").str.contains(";", literal=True)),
                        do_unique=True,
                    ),
                    entry_name,
                    f"{quant}",
                )
                quant_data.calc_cv(cond="all", min_reps=3, temp_reverse_log_scale=2)
                quant_data.calc_cv(cond="all", min_reps=2, temp_reverse_log_scale=2)
                quant_data.count_detected_replicates()
                quant_data.calc_ratio(
                    base_cond="H5_Y100",
                    is_log=True,
                    temp_reverse_log_scale=2,
                    div_method="agg_and_divide",
                    agg_method="mean",
                )

    report.dump()


##这里是需要进行矫正的

##这里进行矫正根据前面算的矫正因子

In [ ]:
import polars as pl
from pathlib import Path
import glob

# ------------------------------
# 路径设置
# ------------------------------
quant_dir = Path("/home/searches/"
                 "example_diann_report/lipana_analysis_orgin")

output_dir = Path("/home/searches/"
                  "example_diann_report/lipana_analysis_corrected")
output_dir.mkdir(exist_ok=True, parents=True)

# mapping 文件：stripped_peptide -> protein_group
mapping_file = quant_dir / "search_report.parquet"
df_mapping = pl.read_parquet(mapping_file).select(["protein_group", "stripped_peptide"])
# 确保 mapping 每个 peptide 只对应一个 protein_group
df_mapping = df_mapping.unique(subset="stripped_peptide")

# 矫正因子文件：protein_group -> ratio_H2_Y100_to_H5_Y100
pg_file = Path("/home/searches/"
               "example_diann_report/lipana_analysis/quant!!protein_group!!PG.Quantity.parquet")
df_pg = pl.read_parquet(pg_file).select([
    "protein_group",
    pl.col("ratio_H2_Y100_to_H5_Y100").alias("correction_factor")
])
# NA 视作 0
df_pg = df_pg.fill_null(0)

# ------------------------------
# 批量处理 quant!!stripped_peptide 开头的文件
# ------------------------------
files = glob.glob(str(quant_dir / "quant!!stripped_peptide*.parquet"))

for file_path in files:
    df_pep = pl.read_parquet(file_path)
    
    # 只保留每个 stripped_peptide 一行，避免 join 后重复
    df_pep = df_pep.unique(subset="stripped_peptide")
    
    # 添加 protein_group
    df_pep = df_pep.join(df_mapping, left_on="stripped_peptide", right_on="stripped_peptide", how="left")
    
    # 添加矫正因子
    df_pep = df_pep.join(df_pg, left_on="protein_group", right_on="protein_group", how="left")
    
    # NA 当作 0
    df_pep = df_pep.fill_null(0)
    
    # 对 ratio 列进行矫正（减去矫正因子）
    ratio_cols = [col for col in df_pep.columns if col.startswith("ratio_")]
    for col in ratio_cols:
        df_pep = df_pep.with_columns(
            (pl.col(col) - pl.col("correction_factor")).alias(col)
        )
    
    # 删除矫正因子列
    df_pep = df_pep
    
    # 输出 parquet
    out_file = output_dir / Path(file_path).name.replace(".parquet", "_corrected.parquet")
    df_pep.write_parquet(out_file)
    
    print(f"Processed and saved: {out_file}")